# Notebook 05 – Critical Evaluation & Adversarial Testing (Part E)

Run adversarial queries through both RAG and pure-LLM.
Record manual accuracy/hallucination annotations in `experiment_logs/partE_adversarial.md`.

In [1]:
%pip install -q pandas pypdf numpy sentence-transformers openai tiktoken python-dotenv scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.join('..', 'backend'))
import pandas as pd
from raghana.pipeline import run as rag_run
from raghana.generate import generate_no_retrieval

ADVERSARIAL_QUERIES = [
    {
        'id': 'A1',
        'type': 'Ambiguous',
        'query': 'Who won in Ghana?',
        'ground_truth': 'Ambiguous — must specify year. 2020: Akufo-Addo (NPP). 2016: Akufo-Addo (NPP). 2012: Mahama (NDC).',
    },
    {
        'id': 'A2',
        'type': 'False-premise (misleading)',
        'query': 'How much did the NDC spend on Free SHS in 2024?',
        'ground_truth': 'False premise — Free SHS is an NPP policy. Correct answer: refuse or note false premise.',
    },
    {
        'id': 'A3',
        'type': 'Incomplete',
        'query': 'How many votes were cast in Ashanti?',
        'ground_truth': 'Incomplete — year not specified. Should ask for clarification or list all years.',
    },
    {
        'id': 'A4',
        'type': 'Out-of-domain',
        'query': 'Who is the President of the United States?',
        'ground_truth': 'Out-of-domain. Should refuse with the standard refusal message.',
    },
]

## 5.1 Run adversarial queries — RAG vs Pure LLM

In [3]:
records = []
for q_info in ADVERSARIAL_QUERIES:
    q = q_info['query']
    print(f"\n{'='*70}")
    print(f"[{q_info['id']}] {q_info['type']}")
    print(f"Query: {q}")
    print(f"Ground truth: {q_info['ground_truth']}")

    # RAG answer
    rag_result = rag_run(q)
    print(f"\nRAG answer:\n{rag_result.answer}")

    # Pure-LLM answer
    pure_answer = generate_no_retrieval(q)
    print(f"\nPure-LLM answer:\n{pure_answer}")

    records.append({
        'id': q_info['id'],
        'type': q_info['type'],
        'query': q,
        'ground_truth': q_info['ground_truth'],
        'rag_answer': rag_result.answer,
        'pure_llm_answer': pure_answer,
        # Manual annotations — fill these in after reviewing outputs
        'rag_correct': None,        # True / False
        'rag_hallucinates': None,   # True / False
        'pure_correct': None,
        'pure_hallucinates': None,
    })

df_results = pd.DataFrame(records)
df_results.to_csv('experiment_logs/adversarial_results.csv', index=False)
print('\nResults saved to experiment_logs/adversarial_results.csv')

2026-04-24 16:24:12,391 [INFO] raghana: Use pytorch device_name: cpu


2026-04-24 16:24:12,392 [INFO] raghana: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2



[A1] Ambiguous
Query: Who won in Ghana?
Ground truth: Ambiguous — must specify year. 2020: Akufo-Addo (NPP). 2016: Akufo-Addo (NPP). 2012: Mahama (NDC).


2026-04-24 16:24:19,145 [INFO] raghana: [retrieve] 6776ms | 5 chunks | top_score=0.030 | mode=hybrid


2026-04-24 16:24:19,153 [INFO] raghana: [prompt_build] 5ms | 8700 chars


...

2026-04-24 16:24:28,759 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:24:28,766 [INFO] raghana: [generate] 1730ms | I do not have that information in the provided documents.



RAG answer:
I do not have that information in the provided documents.


2026-04-24 16:24:30,314 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Pure-LLM answer:
As of my last knowledge update in October 2023, the President of the United States is Joe Biden. He took office on January 20, 2021. Please verify with up-to-date sources, as political positions can change.

Results saved to experiment_logs/adversarial_results.csv


## 5.2 Consistency test — run each query 3×

Measure sentence-level Jaccard similarity across 3 runs to quantify consistency.

In [4]:
import re

def sentence_tokens(text):
    sentences = re.split(r'[.!?]+', text)
    return {frozenset(s.lower().split()) for s in sentences if s.strip()}

def jaccard(a, b):
    all_sents = a | b
    if not all_sents:
        return 1.0
    intersection = sum(1 for s in all_sents if s in a and s in b)
    return intersection / len(all_sents)

for q_info in ADVERSARIAL_QUERIES[:2]:  # Run on first 2 to save API calls
    q = q_info['query']
    runs = [rag_run(q).answer for _ in range(3)]
    j12 = jaccard(sentence_tokens(runs[0]), sentence_tokens(runs[1]))
    j13 = jaccard(sentence_tokens(runs[0]), sentence_tokens(runs[2]))
    j23 = jaccard(sentence_tokens(runs[1]), sentence_tokens(runs[2]))
    avg_j = (j12 + j13 + j23) / 3
    print(f'Query: {q[:50]}...')
    print(f'  Avg Jaccard consistency: {avg_j:.3f}\n')

2026-04-24 16:24:30,379 [INFO] raghana: [retrieve] 28ms | 5 chunks | top_score=0.030 | mode=hybrid


2026-04-24 16:24:30,383 [INFO] raghana: [prompt_build] 3ms | 8700 chars


2026-04-24 16:24:32,124 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:24:32,126 [INFO] raghana: [generate] 1741ms | I do not have that information in the provided documents.


2026-04-24 16:24:32,161 [INFO] raghana: [retrieve] 33ms | 5 chunks | top_score=0.030 | mode=hybrid


...

2026-04-24 16:24:35,501 [INFO] raghana: [retrieve] 34ms | 5 chunks | top_score=0.016 | mode=hybrid


2026-04-24 16:24:35,505 [INFO] raghana: [prompt_build] 3ms | 5390 chars


2026-04-24 16:24:36,685 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:24:36,687 [INFO] raghana: [generate] 1180ms | I do not have that information in the provided documents.


Query: How much did the NDC spend on Free SHS in 2024?...
  Avg Jaccard consistency: 1.000



## 5.3 Summary table (fill in manually after running)

| ID | Type | RAG correct | RAG hallucinates | Pure-LLM correct | Pure-LLM hallucinates |
|---|---|---|---|---|---|
| A1 | Ambiguous | ? | ? | ? | ? |
| A2 | False-premise | ? | ? | ? | ? |
| A3 | Incomplete | ? | ? | ? | ? |
| A4 | Out-of-domain | ? | ? | ? | ? |

**Key finding to log:** Does RAG refuse correctly on A2 and A4?
Does pure-LLM invent an answer for A2 (claiming NDC spent on Free SHS)?

**Manual log:** `experiment_logs/partE_adversarial.md`

## 5.4 Part G — Multi-step reasoning accuracy

Compare standard RAG vs multi-step for numeric queries.

In [5]:
NUMERIC_QUERIES = [
    {'query': 'Total NPP votes in Ashanti Region 2020',
     'ground_truth_hint': '1,795,824'},
    {'query': 'Compare NPP and NDC votes in Greater Accra 2020',
     'ground_truth_hint': 'Numerical comparison'},
    {'query': 'Who got the highest votes in any single region in 2020?',
     'ground_truth_hint': 'NPP Ashanti 1,795,824'},
    {'query': 'Total NDC votes in 2020',
     'ground_truth_hint': 'Sum across all regions'},
    {'query': 'What percentage of Ashanti Region votes did NPP receive in 2020?',
     'ground_truth_hint': '72.79%'},
]

print('=== Standard RAG vs Multi-step RAG for numeric queries ===')
for nq in NUMERIC_QUERIES:
    r = rag_run(nq['query'], mode='hybrid')
    is_multistep = r.multistep_trace is not None
    print(f"\nQ: {nq['query']}")
    print(f"Ground truth hint: {nq['ground_truth_hint']}")
    print(f"Multi-step triggered: {is_multistep}")
    if is_multistep:
        print(f"Computed: {r.multistep_trace['computed_result']}")
    print(f"Answer: {r.answer[:200]}")
    print('-' * 60)

2026-04-24 16:24:36,729 [INFO] raghana: [retrieve] 33ms | 5 chunks | top_score=0.032 | mode=hybrid


2026-04-24 16:24:36,731 [INFO] raghana: [multistep] 0ms | Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020


2026-04-24 16:24:36,734 [INFO] raghana: [prompt_build] 1ms | 2776 chars


=== Standard RAG vs Multi-step RAG for numeric queries ===


2026-04-24 16:24:37,658 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

2026-04-24 16:24:41,633 [INFO] raghana: [prompt_build] 1ms | 2802 chars



Q: Total NDC votes in 2020
Ground truth hint: Sum across all regions
Multi-step triggered: True
Computed: Total NDC votes across all retrieved regions/years (2020): 1,127,384
Answer: I do not have that information in the provided documents.
------------------------------------------------------------


2026-04-24 16:24:42,973 [INFO] raghana: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


2026-04-24 16:24:43,086 [INFO] raghana: [generate] 1452ms | The NPP received 72.79% of the votes in the 2020 presidential election in Ashant



Q: What percentage of Ashanti Region votes did NPP receive in 2020?
Ground truth hint: 72.79%
Multi-step triggered: True
Computed: Total NPP votes across all retrieved regions/years (1992, 1996, 2000, 2008, 2020): 6,266,614
Answer: The NPP received 72.79% of the votes in the 2020 presidential election in Ashanti Region [C1].
------------------------------------------------------------
